In [4]:
!nvidia-smi

Sat Mar 21 16:48:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!apt-get install -y nsight-systems-2025.5.2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nsight-systems-2025.5.2 is already the newest version (2025.5.2.266-255236693005v0).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [2]:
%%writefile add.cu
#include <cmath>
#include <stdio.h>
#include <stdlib.h>

__global__ void vecAddKernel(float *A, float *B, float *C, int n);

int main() {
  int n = 1 << 20; // 1M elements
  int size = n * sizeof(float);

  float *A_h = (float*) malloc(size);
  float *B_h = (float*) malloc(size);
  float *C_h = (float*) malloc(size);

  for (int i = 0; i < n; ++i){
    A_h[i] = 1.0f;
    B_h[i] = 2.0f;
  }

  float *A_d, *B_d, *C_d;

  // Part 1: Allocate device memory for A, B, and C
  // Copy A and B to device memory

  cudaMalloc((void **)&A_d, size);
  cudaMalloc((void **)&B_d, size);
  cudaMalloc((void **)&C_d, size);

  cudaMemcpy(A_d, A_h, size, cudaMemcpyHostToDevice);
  cudaMemcpy(B_d, B_h, size, cudaMemcpyHostToDevice);

  // Part 2: Call kernel – to launch a grid of threads
  // to perform the actual vector addition
  vecAddKernel<<<ceil(n / 256.0), 256>>>(A_d, B_d, C_d, n);

  // Part 3: Copy C from the device memory
  cudaMemcpy(C_h, C_d, size, cudaMemcpyDeviceToHost);

  // Free device vectors
  cudaFree(A_d);
  cudaFree(B_d);
  cudaFree(C_d);

  printf("Printing the first 20 results\n");
  for (int i = 0; i < 20; ++i){
    printf("C_h at %i: %f\n", i, C_h[i]);
  }

  free(A_h);
  free(B_h);
  free(C_h);

  return 0;
}

__global__ void vecAddKernel(float *A, float *B, float *C, int n) {
  int global_i = blockDim.x * blockIdx.x + threadIdx.x;
  if (global_i < n) {
    C[global_i] = A[global_i] + B[global_i];
  }
  return;
}

Overwriting add.cu


In [3]:
!nvcc add.cu -o add
!./add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Printing the first 20 results
C_h at 0: 3.000000
C_h at 1: 3.000000
C_h at 2: 3.000000
C_h at 3: 3.000000
C_h at 4: 3.000000
C_h at 5: 3.000000
C_h at 6: 3.000000
C_h at 7: 3.000000
C_h at 8: 3.000000
C_h at 9: 3.000000
C_h at 10: 3.000000
C_h at 11: 3.000000
C_h at 12: 3.000000
C_h at 13: 3.000000
C_h at 14: 3.000000
C_h at 15: 3.000000
C_h at 16: 3.000000
C_h at 17: 3.000000
C_h at 18: 3.000000
C_h at 19: 3.000000


In [5]:
!nsys profile --stats=true ./add

Printing the first 20 results
C_h at 0: 3.000000
C_h at 1: 3.000000
C_h at 2: 3.000000
C_h at 3: 3.000000
C_h at 4: 3.000000
C_h at 5: 3.000000
C_h at 6: 3.000000
C_h at 7: 3.000000
C_h at 8: 3.000000
C_h at 9: 3.000000
C_h at 10: 3.000000
C_h at 11: 3.000000
C_h at 12: 3.000000
C_h at 13: 3.000000
C_h at 14: 3.000000
C_h at 15: 3.000000
C_h at 16: 3.000000
C_h at 17: 3.000000
C_h at 18: 3.000000
C_h at 19: 3.000000
Generating '/tmp/nsys-report-3e74.qdstrm'
[1/8] [========================100%] report2.nsys-rep
Processing 1183 events:        ] report2.sqlite
[2/8] [========================100%] report2.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /content/report2.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  --------  -----------  ------

In [6]:
# TODO(human): Run ncu (Nsight Compute) to get kernel-level profiling.
# See the guidance in the chat for details.

!ncu ./add --target-processes all

==PROF== Connected to process 17052 (/content/add)
==PROF== Profiling "vecAddKernel" - 0: 0%....50%....100% - 9 passes
Printing the first 20 results
C_h at 0: 3.000000
C_h at 1: 3.000000
C_h at 2: 3.000000
C_h at 3: 3.000000
C_h at 4: 3.000000
C_h at 5: 3.000000
C_h at 6: 3.000000
C_h at 7: 3.000000
C_h at 8: 3.000000
C_h at 9: 3.000000
C_h at 10: 3.000000
C_h at 11: 3.000000
C_h at 12: 3.000000
C_h at 13: 3.000000
C_h at 14: 3.000000
C_h at 15: 3.000000
C_h at 16: 3.000000
C_h at 17: 3.000000
C_h at 18: 3.000000
C_h at 19: 3.000000
==PROF== Disconnected from process 17052
[17052] add@127.0.0.1
  vecAddKernel(float *, float *, float *, int) (4096, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.64
    SM Frequency                